# Task_Classification_Procedure

Step 1 — Load the locked task universe

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import json
from google.colab import userdata
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score


ROOT = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full/02_Task_Data_and_Classification")

TASK_UNIVERSE_FILE = (
    ROOT / "Output" / "Occupational_Task_Universe_ONET30_v2.csv"
)

task_universe = pd.read_csv(TASK_UNIVERSE_FILE)

#display(task_universe.head())

Step 2 — confirm the locked universe and unique task IDs.

In [ ]:
print(f"Tasks: {len(task_universe):,}")
print(f"Unique Task IDs: {task_universe['task_id'].nunique():,}")
print(f"Occupations: {task_universe['soc6'].nunique()}")

Tasks: 16,486
Unique Task IDs: 16,486
Occupations: 771


In [ ]:
task_universe.groupby("task_source_type")["soc6"].nunique()

,soc6
task_source_type,
direct,748
recovered_child,23


Step 3: finalize the classification rubric and prompt.

| Label                           | Decision rule                                                                                                                                                       |
| ------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Limited GenAI applicability** | GenAI cannot materially perform the core task or provide meaningful assistance to the central work outcome.                                                         |
| **Augmentation-oriented**       | GenAI can materially assist with the task, but substantive human judgment, action, interaction, or expertise remains necessary to produce the central work outcome. |
| **Automation-oriented**         | GenAI can perform most of the core task, with the human role mainly limited to initiating, monitoring, approving, or handling exceptions.                           |
| **Indeterminate mechanism**     | The task statement does not provide enough information to distinguish the mechanism reliably, or multiple mechanisms are equally central.                           |


In [ ]:
RUBRIC_VERSION = "v6"

SYSTEM_PROMPT = """
You are classifying an occupational task according to the dominant
Generative AI task-affordance mechanism.

Assess the task using generally available Generative AI capabilities that
operate on digital information and content. Do not assume robotics, physical
actuation, or specialized non-GenAI automation.

Physical execution is outside the GenAI capability scope. If the central work
activity is primarily physical, GenAI support through information retrieval,
preparation, drafting, documentation, or other surrounding digital activities
does not by itself constitute augmentation. Classify such a task as Limited
GenAI applicability unless GenAI can directly perform a substantive
non-physical component explicitly contained in the task as stated.

The classification is based on two principles:
- Automation: GenAI performs or takes over most of the substantive task activity.
- Augmentation: GenAI and the human jointly perform substantive components of
  the task.

Classify the TASK AS STATED, focusing on its central work activity rather than
activities that merely surround or support it.

Use exactly one label:

1. Limited GenAI applicability
   GenAI cannot directly perform a substantive component of the central work
   activity. Assistance limited to peripheral preparation, information
   retrieval, drafting, scheduling, documentation, or general support does not
   qualify as augmentation when those activities are not themselves part of
   the central task.

2. Augmentation-oriented
   GenAI can directly perform a substantive component of the central work
   activity, but meaningful human contribution remains necessary to complete
   the task. The remaining human contribution may involve judgment, expertise,
   interpersonal interaction, accountability, decision-making, or execution of
   other substantive task components.

3. Automation-oriented
   GenAI can perform most of the substantive central work activity such that
   the central task outcome can largely be produced by GenAI. Human involvement
   is primarily initiation, monitoring, verification, approval, or exception
   handling.

4. Indeterminate mechanism
   The task statement does not provide sufficient information to determine the
   dominant mechanism reliably, or two or more classifications remain similarly
   plausible after applying the decision rules.

Decision rules:
- Evaluate the central work activity explicitly described in the task statement.
- GenAI must directly perform part of the stated task for the task to qualify
  as augmentation or automation.
- For Augmentation-oriented or Automation-oriented classification, GenAI must
  be capable of directly executing a substantive part of the central action
  expressed by the task statement, not merely improving, informing, preparing
  for, or supporting the human who performs that action.
- For tasks whose central action is inherently relational, interpersonal,
  institutional, authoritative, or physical, digital analysis or content
  generation surrounding that action does not by itself satisfy the
  direct-performance requirement.
- Useful assistance somewhere in the surrounding workflow is not sufficient.
- If GenAI only supports activities peripheral to the central task ->
  Limited GenAI applicability.
- If GenAI directly performs a substantive part of the central task while the
  human remains materially involved -> Augmentation-oriented.
- If GenAI can perform most of the substantive central task ->
  Automation-oriented.
- If the dominant mechanism cannot be determined reliably, or two or more
  classifications remain similarly plausible after applying the decision
  rules -> Indeterminate mechanism.

Preparation, information retrieval, drafting, scheduling, and documentation
are considered peripheral only when they support a different central work
activity. If one of these activities is itself the central activity explicitly
stated in the task, direct GenAI performance of that activity counts toward
augmentation or automation.

Do not infer additional task components that are not stated or reasonably
implied by the occupational task statement.

Ambiguity rule:
- Set ambiguity_flag = True only when the task statement is materially
  underspecified or when two or more classifications remain similarly plausible
  after applying the decision rules.
- "Materially underspecified" means the task statement omits information needed
  to apply the decision rules and choose a label. For example, it may name an
  activity without indicating whether GenAI or the human performs the
  substantive component. It does not mean that a task is merely brief.
- Indeterminate mechanism must always have ambiguity_flag = True.
- Otherwise set ambiguity_flag = False.

Return:
- one label,
- a concise rationale explaining how GenAI relates to the central work activity,
- an ambiguity flag.
"""

In [ ]:
def build_task_prompt(row):
    return f"""
Occupation: {row['occupation_title']}
Task: {row['task_statement']}
"""

Step 4 — Configure the model

In [ ]:
api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Add it to Colab Secrets and enable notebook access."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.6-luna"
REASONING_EFFORT = "low"

In [ ]:
OUTPUT_SCHEMA = {
    "type": "json_schema",
    "name": "task_classification",
    "schema": {
        "type": "object",
        "properties": {
            "label": {
                "type": "string",
                "enum": [
                    "Limited GenAI applicability",
                    "Augmentation-oriented",
                    "Automation-oriented",
                    "Indeterminate mechanism"
                ]
            },
            "rationale": {
                "type": "string"
            },
            "ambiguity_flag": {
                "type": "boolean"
            }
        },
        "required": ["label", "rationale", "ambiguity_flag"],
        "additionalProperties": False
    },
    "strict": True
}

Step 5 — Run Pass 1 and Pass 2

Each pass uses its own checkpoint and resumes from its saved task IDs.

In [ ]:
PASS1_CHECKPOINT = Path(
   ROOT / "Output" /
    "Occupational_Task_Classification_Pass1_v6.jsonl"
)
PASS2_CHECKPOINT = Path(
    ROOT / "Output" /
    "Occupational_Task_Classification_Pass2_v6.jsonl"
)

In [ ]:
for pass_number, checkpoint in enumerate((PASS1_CHECKPOINT, PASS2_CHECKPOINT), start=1):
    completed = set()
    if checkpoint.exists():
        with checkpoint.open("r") as f:
            completed = {json.loads(line)["task_id"] for line in f if line.strip()}

    remaining = task_universe.loc[~task_universe["task_id"].isin(completed)]
    print(f"Pass {pass_number}: {len(completed):,} task IDs saved, {len(remaining):,} remaining.")

    for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc=f"Pass {pass_number}"):
        task_id = int(row["task_id"])
        if task_id in completed:
            continue

        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": REASONING_EFFORT},
            text={"format": OUTPUT_SCHEMA},
            input=[
                {"role": "developer", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_task_prompt(row)}
            ]
        )

        result = json.loads(response.output_text)

        output = {
            "task_id": task_id,
            f"pass{pass_number}_label": result["label"],
            f"pass{pass_number}_rationale": result["rationale"],
            f"pass{pass_number}_ambiguity_flag": result["ambiguity_flag"],
            f"pass{pass_number}_response_id": response.id
        }

        with open(checkpoint, "a") as f:
            f.write(json.dumps(output) + "\n")

        completed.add(task_id)

    saved = pd.read_json(checkpoint, lines=True)
    print(
        f"Pass {pass_number}: {len(saved):,} saved rows, "
        f"{saved['task_id'].nunique():,} unique task IDs, "
        f"{saved['task_id'].duplicated().sum():,} duplicate task IDs."
    )

Step 6 — compare Pass 1 vs. Pass 2

In [ ]:
pass1 = pd.read_json(PASS1_CHECKPOINT, lines=True)
pass2 = pd.read_json(PASS2_CHECKPOINT, lines=True)

comparison = pass1.merge(
    pass2,
    on="task_id",
    how="inner"
)

comparison["agreement_flag"] = (
    comparison["pass1_label"] == comparison["pass2_label"]
)

print(f"Matched tasks: {len(comparison):,}")
print(f"Agreement: {comparison['agreement_flag'].sum():,}")
print(f"Disagreement: {(~comparison['agreement_flag']).sum():,}")
print(f"Agreement rate: {comparison['agreement_flag'].mean():.1%}")

Matched tasks: 16,486
Agreement: 15,094
Disagreement: 1,392
Agreement rate: 91.6%


In [ ]:
excluded_tasks = pass1["pass1_label"].eq("Indeterminate mechanism").sum()
print("Task records excluded:", excluded_tasks)
print("Task records retained:", len(pass1) - excluded_tasks)

Task records excluded: 41
Task records retained: 16445


In [ ]:
display(
    pd.crosstab(
        comparison["pass1_label"],
        comparison["pass2_label"],
        margins=True
    )
)

pass2_label,Augmentation-oriented,Automation-oriented,Indeterminate mechanism,Limited GenAI applicability,All
pass1_label,,,,,
Augmentation-oriented,6671,195,18,440,7324
Automation-oriented,210,533,0,8,751
Indeterminate mechanism,15,1,18,7,41
Limited GenAI applicability,484,6,8,7872,8370
All,7380,735,44,8327,16486


In [ ]:
kappa = cohen_kappa_score(
    comparison["pass1_label"],
    comparison["pass2_label"]
)

print(f"Repeated-run Cohen's kappa: {kappa:.3f}")

Repeated-run Cohen's kappa: 0.844


Step 7

Pass 1 = provisional label

Pass 2 = reproducibility check

In [ ]:
# Step 7 — Create provisional classification file from Pass 1 only

pass1 = pd.read_json(PASS1_CHECKPOINT, lines=True)

provisional = task_universe.merge(
    pass1[["task_id", "pass1_label"]],
    on="task_id",
    how="left"
)

provisional = provisional.rename(
    columns={"pass1_label": "provisional_label"}
)

# Classification metadata
provisional["rubric_version"] = RUBRIC_VERSION
provisional["model"] = MODEL
provisional["reasoning_effort"] = REASONING_EFFORT

# Integrity check
print(f"Tasks: {len(provisional):,}")
print(f"Unique task IDs: {provisional['task_id'].nunique():,}")
print(f"Missing labels: {provisional['provisional_label'].isna().sum()}")

display(
    provisional[
        [
            "task_id",
            "occupation_title",
            "task_statement",
            "provisional_label"
        ]
    ].head()
)

In [ ]:
PROVISIONAL_FILE = Path(
    ROOT / "Output" /
    "Occupational_Task_Classification_Provisional_v6.csv"
)

if PROVISIONAL_FILE.exists():
    print("Provisional v6 file already exists — file not overwritten.")
else:
    provisional.to_csv(PROVISIONAL_FILE, index=False)
    print(f"Saved and locked: {PROVISIONAL_FILE}")